# 03 — Hybrid RAG eval + optional publish

Phase 5 path: **clone → corpus → index → retrieval metrics → optional train → grounded generate eval → publish**.

**Defaults are dry-run / CPU-safe.** Do not flip `RUN_GPU` unless you are on a Kaggle T4 (or similar) and intend to download embed weights / load Unsloth.

Do **not** copy fixture Recall@k / nDCG / citation-hit percentages onto a resume. Those numbers live in `evals/reports/` after a real `--run`.

## 0. Flags (dry-run by default)

In [1]:
from pathlib import Path
import os
import sys

# Flip only on Kaggle GPU after you have accepted the extra download / train time.
RUN_GPU = True
RUN_TRAIN = True          # only if you need a short 20-step adapter smoke
RUN_GENERATE_EVAL = True
PUBLISH_ADAPTER = True    # only after adapter exists + HF_TOKEN secret is set   # HF Hub upload; needs HF_TOKEN secret + adapter dir

IN_KAGGLE = Path("/kaggle").exists()
print("IN_KAGGLE:", IN_KAGGLE)
print("RUN_GPU:", RUN_GPU, "RUN_TRAIN:", RUN_TRAIN, "RUN_GENERATE_EVAL:", RUN_GENERATE_EVAL, "PUBLISH_ADAPTER:", PUBLISH_ADAPTER)

IN_KAGGLE: True
RUN_GPU: True RUN_TRAIN: True RUN_GENERATE_EVAL: True PUBLISH_ADAPTER: True


## 1. Clone / path setup

In [2]:
# If this notebook lives in the repo, use ../src. On a fresh Kaggle kernel, clone first.
REPO = Path("..").resolve()
if not (REPO / "src" / "earnings_call_research_assistant").exists():
    if IN_KAGGLE:
        !git clone https://github.com/nuwanda94/earnings-call-research-assistant.git
        REPO = Path("earnings-call-research-assistant").resolve()
    else:
        raise FileNotFoundError("Cannot find package src; run from notebooks/ or clone the repo.")

src = (REPO / "src").resolve()
if str(src) not in sys.path:
    sys.path.insert(0, str(src))
os.chdir(REPO)
print("cwd:", Path.cwd())
print("src:", src)

Cloning into 'earnings-call-research-assistant'...
remote: Enumerating objects: 508, done.
remote: Counting objects: 100% (93/93), done.
remote: Compressing objects: 100% (91/91), done.
remote: Total 508 (delta 47), reused 0 (delta 0), pack-reused 415 (from 1)
Receiving objects: 100% (508/508), 210.60 KiB | 5.26 MiB/s, done.
Resolving deltas: 100% (287/287), done.
cwd: /kaggle/working/earnings-call-research-assistant
src: /kaggle/working/earnings-call-research-assistant/src


In [3]:
if IN_KAGGLE:
    %pip install -q pyyaml
    if RUN_GPU:
        %pip install -q unsloth transformers accelerate bitsandbytes sentence-transformers

Note: you may need to restart the kernel to use updated packages.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.4/75.4 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 22.4/22.4 MB 65.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 86.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 43.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 31.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.8/73.8 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 29.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 67.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 90.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 59.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.9/216.9 kB 14.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.7/119.7 kB 10.1 M

## 2. Corpus (defines N)

In [4]:
!python scripts/build_rag_corpus.py

corpus_version: v0.1.0
N (n_chunks): 4
n_documents: 3
sources: {"earnings_transcripts": 2, "finance_alpaca": 1, "fiqa": 1}
Wrote /kaggle/working/earnings-call-research-assistant/data/rag/corpus_v0.1.0/chunks.jsonl
Wrote /kaggle/working/earnings-call-research-assistant/data/rag/corpus_v0.1.0/manifest.json


In [5]:
import json
from pathlib import Path

manifest_path = Path("data/rag/corpus_v0.1.0/manifest.json")
if manifest_path.is_file():
    manifest = json.loads(manifest_path.read_text())
    print("N chunks:", manifest.get("n_chunks"))
    print("n_documents:", manifest.get("n_documents"))
    print("version:", manifest.get("version"))
    print("sources:", manifest.get("sources"))
else:
    print("manifest missing — corpus script should have written it")

N chunks: 4
n_documents: 3
version: v0.1.0
sources: {'earnings_transcripts': 2, 'finance_alpaca': 1, 'fiqa': 1}


## 3. Hybrid index (BM25 + dense)

Dry-run uses hashed bag-of-tokens embeddings (offline). `RUN_GPU` rebuilds dense with `sentence-transformers`.

In [6]:
if RUN_GPU:
    !python scripts/build_rag_index.py --run --query "operating margin guidance" --k 5
else:
    !python scripts/build_rag_index.py --query "operating margin guidance" --k 3

backend: bm25
n_docs: 4
avgdl: 32.75
corpus_version: v0.1.0
Wrote /kaggle/working/earnings-call-research-assistant/data/rag/indices/bm25/index.json
Wrote /kaggle/working/earnings-call-research-assistant/data/rag/indices/bm25/meta.json
Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).
config_sentence_transformers.json: 100%|████████| 116/116 [00:00<00:00, 586kB/s]
README.md: 10.5kB [00:00, 25.0MB/s]
sentence_bert_config.json: 100%|██████████████| 53.0/53.0 [00:00<00:00, 262kB/s]
config.json: 100%|█████████████████████████████| 612/612 [00:00<00:00, 4.09MB/s]
model.safetensors: 100%|███████████████████| 90.9M/90.9M [00:02<00:00, 44.6MB/s]
Loading weights: 100%|██████████████████████| 103/103 [00:00<00:00, 3014.89it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

No

## 4. Gold eval set (seed 3407)

In [7]:
!python scripts/build_rag_eval_set.py

eval_set_version: v0.1.0
seed: 3407
n_queries: 12
corpus_n_chunks: 4
n_unique_gold_chunks: 4
excluded_train_pairs: 0
Wrote /kaggle/working/earnings-call-research-assistant/evals/rag_eval_set.jsonl
Wrote /kaggle/working/earnings-call-research-assistant/evals/rag_eval_set.manifest.json
{"notes": "Seed=3407. Sampled 12 / 12 grounded candidates. Every gold_chunk_id is in corpus /kaggle/working/earnings-call-research-assistant/data/rag/corpus_v0.1.0 (N=4). Excluded 0 train-split pair_ids. v0.1 fixture corpora yield fewer than 30–50 queries; grow when N scales."}


## 5. Retrieval metrics — Recall@k / nDCG@k

In [8]:
if RUN_GPU:
    !python scripts/eval_retrieval.py --run
else:
    !python scripts/eval_retrieval.py

Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).
Loading weights: 100%|██████████████████████| 103/103 [00:00<00:00, 3013.36it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|██████████████████████| 103/103 [00:00<00:00, 2894.12it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|██████████████████████| 103/103 [00:00<00:00, 2927.20it/s

In [9]:
metrics_path = Path("evals/reports/rag_metrics.json")
if metrics_path.is_file():
    payload = json.loads(metrics_path.read_text())
    print("n_queries:", payload.get("n_queries"))
    print("n_index_docs:", payload.get("n_index_docs"))
    print("dense_backend:", payload.get("dense_backend"))
    for name, report in (payload.get("backends") or {}).items():
        print(name, "recall", report.get("mean_recall"), "ndcg", report.get("mean_ndcg"))
    print("note: fixture / dry-run scores are not resume numbers")
else:
    print("rag_metrics.json not written")

n_queries: 12
n_index_docs: 4
dense_backend: sentence-transformers
bm25 recall {'@1': 0.5, '@3': 0.75, '@5': 1.0, '@10': 1.0} ndcg {'@1': 0.5, '@3': 0.625, '@5': 0.732669, '@10': 0.732669}
dense recall {'@1': 0.416667, '@3': 0.833333, '@5': 1.0, '@10': 1.0} ndcg {'@1': 0.416667, '@3': 0.646822, '@5': 0.718601, '@10': 0.718601}
hybrid recall {'@1': 0.416667, '@3': 0.833333, '@5': 1.0, '@10': 1.0} ndcg {'@1': 0.416667, '@3': 0.646822, '@5': 0.718601, '@10': 0.718601}
note: fixture / dry-run scores are not resume numbers


## 6. Optional short SFT (off by default)

Leave `RUN_TRAIN=False` in automation and most sessions. On Kaggle T4 set both `RUN_GPU` and `RUN_TRAIN` for a 20-step smoke.

In [10]:
if RUN_TRAIN and RUN_GPU:
    !python scripts/train_sft.py --run --max-steps 20
else:
    !python scripts/train_sft.py
    print("SFT dry-run only (no weights). Set RUN_TRAIN and RUN_GPU for a short GPU smoke.")

Traceback (most recent call last):
  File "/kaggle/working/earnings-call-research-assistant/scripts/train_sft.py", line 131, in <module>
    raise SystemExit(main())
                     ^^^^^^
  File "/kaggle/working/earnings-call-research-assistant/scripts/train_sft.py", line 104, in main
    plan = run_sft(
           ^^^^^^^^
  File "/kaggle/working/earnings-call-research-assistant/src/earnings_call_research_assistant/training/sft.py", line 413, in run_sft
    splits = load_sft_splits(run.dataset_dir, require_train=require_train and not dry_run)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/kaggle/working/earnings-call-research-assistant/src/earnings_call_research_assistant/training/sft.py", line 233, in load_sft_splits
    raise FileNotFoundError(
FileNotFoundError: No training rows under /kaggle/working/earnings-call-research-assistant/data/processed/ecra-sft-v0.1.0. Run `python scripts/select_dataset.py` first (expected /kagg

## 7. Grounded generate eval (citation-hit + token F1)

Dry-run writes placeholders. `--run` loads `InferenceHarness` (base + optional adapter).

In [11]:
ADAPTER = Path("outputs/adapters/llama32-3b-ecra-sft")
if RUN_GENERATE_EVAL and RUN_GPU:
    if ADAPTER.is_dir():
        !python scripts/eval_rag_generate.py --run --adapter-dir outputs/adapters/llama32-3b-ecra-sft
    else:
        !python scripts/eval_rag_generate.py --run
        print("adapter dir missing; scored base only")
else:
    !python scripts/eval_rag_generate.py

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
INFO NumExpr defaulting to 4 threads.
INFO Disabling Tensorflow because USE_TORCH is set
INFO JAX version 0.7.2 available.
🦥 Unsloth Zoo will now patch everything to make training faster!
INFO HTTP Request: HEAD https://huggingface.co/unsloth/llama-3.2-3b-instruct-unsloth-bnb-4bit/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
INFO HTTP Request: HEAD https://huggingface.co/unsloth/Llama-3.2-3B-Instruct-unsloth-bnb-4bit/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
INFO HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/unsloth/Llama-3.2-3B-Instruct-unsloth-bnb-4bit/19846d3f624f3eb96f3bdd275620c6bc7e21e1f8/config.json "HTTP/1.1 200 OK"
INFO HTTP Request: GET https://huggingface.co/api/resolve-cache/models/unsloth/Llama-3.2-3B-Instruct-unsloth-bnb-4bit/19846d3f624f3eb96f3bdd275620c6bc7e21e1f8/config.json "HTTP/1.1 200 OK"
INFO HTTP Request: HEAD https://huggingface.co/unsloth

In [12]:
gen_path = Path("evals/reports/rag_generation_metrics.json")
if gen_path.is_file():
    gen = json.loads(gen_path.read_text())
    print("dry_run:", gen.get("dry_run"))
    print("n_queries:", gen.get("n_queries"))
    print("top_k:", gen.get("top_k"))
    print("adapter_dir:", gen.get("adapter_dir"))
    print("aggregate:", json.dumps(gen.get("aggregate"), indent=2))
    print(gen.get("note"))
else:
    print("rag_generation_metrics.json not written")

dry_run: False
n_queries: 12
top_k: 5
adapter_dir: None
aggregate: {
  "base": {
    "citation_hit_rate": 0.5,
    "mean_token_f1_vs_context": 0.377492,
    "mean_token_f1_vs_gold": 0.333225,
    "grounded_answer_accuracy": 0.583333
  },
  "adapter": {
    "citation_hit_rate": 0.083325,
    "mean_token_f1_vs_context": 0.0175,
    "mean_token_f1_vs_gold": 0.008325,
    "grounded_answer_accuracy": 0.0
  }
}
Dry-run placeholders score near zero by design. Fill generations with --run on Kaggle before quoting grounded_answer_accuracy.


## 8. Publish adapter (optional, token via env only)

Never paste `HF_TOKEN` into a cell. On Kaggle: Add-ons → Secrets → `HF_TOKEN`.

In [13]:
if IN_KAGGLE:
    try:
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret("HF_TOKEN")
        if token:
            os.environ["HF_TOKEN"] = token
            print("HF_TOKEN loaded from Kaggle secrets (value not printed)")
        else:
            print("HF_TOKEN secret empty")
    except Exception as exc:
        print("no Kaggle secret:", type(exc).__name__)

token_set = bool(os.environ.get("HF_TOKEN") or os.environ.get("HUGGING_FACE_HUB_TOKEN"))
print("token_env_set:", token_set)

if PUBLISH_ADAPTER and token_set and ADAPTER.is_dir():
    !python scripts/publish_adapter.py --repo-id nuwanda94/llama32-3b-ecra-sft --run
else:
    !python scripts/publish_adapter.py
    print("publish dry-run only")

HF_TOKEN loaded from Kaggle secrets (value not printed)
token_env_set: True
INFO Wrote publish plan to outputs/publish_plan.json
dry_run=True repo=skaran786/llama32-3b-ecra-sft adapter=/kaggle/working/earnings-call-research-assistant/outputs/adapters/llama32-3b-ecra-sft exists=False token_present=True uploaded=False
{
  "dry_run": true,
  "adapter_dir": "/kaggle/working/earnings-call-research-assistant/outputs/adapters/llama32-3b-ecra-sft",
  "repo_id": "skaran786/llama32-3b-ecra-sft",
  "private": false,
  "commit_message": "feat: upload ECRA QLoRA adapter",
  "adapter_exists": false,
  "looks_like_adapter": false,
  "token_present": true,
  "token_env": "HF_TOKEN",
  "files": [],
  "uploaded": false,
  "hub_url": null,
  "notes": [
    "Adapter directory missing: /kaggle/working/earnings-call-research-assistant/outputs/adapters/llama32-3b-ecra-sft. Train on Kaggle first (`python scripts/train_sft.py --run`) and copy the folder locally.",
    "Auth: huggingface-cli login  OR  export H

## 9. Done

Artifacts to commit after a **human** Kaggle `--run` (not this automation):

- `data/rag/corpus_v0.1.0/manifest.json` — measured **N**
- `evals/reports/rag_metrics.json` — Recall@k / nDCG@k
- `evals/reports/rag_generation_metrics.json` — citation-hit + token F1

Next automation item: `evals/reports/RAG_EVAL_REPORT.md` + README results + `docs/MODEL_CARD_RAG.md` (Phase 5.8). Fill numbers **only** from those JSON files.